In [52]:
!pip install -q speechbrain


In [53]:
import os, random

random.seed(42)
DATA_ROOT = "/kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian"

all_files = []
for spk in sorted(os.listdir(DATA_ROOT)):
    spk_dir = os.path.join(DATA_ROOT, spk)
    if not os.path.isdir(spk_dir): continue
    for vid in os.listdir(spk_dir):
        vid_dir = os.path.join(spk_dir, vid)
        if not os.path.isdir(vid_dir): continue
        for f in os.listdir(vid_dir):
            if f.endswith('.wav'):
                all_files.append(f"{spk}/{vid}/{f}")

random.shuffle(all_files)
n = len(all_files)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

with open("/kaggle/working/iden_split.txt", "w") as f:
    for i, path in enumerate(all_files):
        if i < n_train:               split_id = 1
        elif i < n_train + n_val:     split_id = 2
        else:                         split_id = 3
        f.write(f"{split_id} {path}\n")

test_files = [all_files[i] for i in range(n_train + n_val, n)]
spk2files = {}
for f in test_files:
    spk = f.split("/")[0]
    spk2files.setdefault(spk, []).append(f)

pos, neg = [], []
spks = list(spk2files.keys())
for spk, files in spk2files.items():
    for i in range(len(files)):
        for j in range(i+1, min(i+4, len(files))):
            pos.append((1, files[i], files[j]))
for i in range(len(spks)):
    for j in range(i+1, len(spks)):
        f1 = random.choice(spk2files[spks[i]])
        f2 = random.choice(spk2files[spks[j]])
        neg.append((0, f1, f2))

k = min(len(pos), len(neg))
trials = random.sample(pos, k) + random.sample(neg, k)
random.shuffle(trials)

with open("/kaggle/working/veri_test.txt", "w") as f:
    for label, p1, p2 in trials:
        f.write(f"{label} {p1} {p2}\n")

print(f"Total: {n} | Train: {n_train} | Val: {n_val} | Test: {n-n_train-n_val}")
print(f"Trial pairs (SV): {len(trials)} | Speakers: {len(spk2files)}")

Total: 4857 | Train: 3399 | Val: 728 | Test: 730
Trial pairs (SV): 552 | Speakers: 24


Create split files

In [54]:
!python train_ecapa.py \
    --data_root /kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian \
    --split_file /kaggle/working/iden_split.txt \
    --save_dir /kaggle/working/checkpoints \
    --epochs 15 --batch_size 64 --lr 1e-3 --num_workers 2

Device: cuda
Train: 3399 | Val: 728 | Speakers: 24
Epoch 01 | train_loss=4.4229 train_acc=0.3659 | val_loss=3.1998 val_acc=0.4437  
  ✓ Saved best (val_acc=0.4437)
Epoch 02 | train_loss=1.5900 train_acc=0.6795 | val_loss=2.7606 val_acc=0.5591  
  ✓ Saved best (val_acc=0.5591)
Epoch 03 | train_loss=0.9995 train_acc=0.7960 | val_loss=3.1087 val_acc=0.5069  
Epoch 04 | train_loss=0.6553 train_acc=0.8485 | val_loss=1.8225 val_acc=0.6607  
  ✓ Saved best (val_acc=0.6607)
Epoch 05 | train_loss=0.3802 train_acc=0.9042 | val_loss=0.7582 val_acc=0.8668  
  ✓ Saved best (val_acc=0.8668)
Epoch 06 | train_loss=0.2393 train_acc=0.9372 | val_loss=0.8172 val_acc=0.8365  
Epoch 07 | train_loss=0.1705 train_acc=0.9517 | val_loss=0.6153 val_acc=0.8695  
  ✓ Saved best (val_acc=0.8695)
Epoch 08 | train_loss=0.1174 train_acc=0.9699 | val_loss=0.6016 val_acc=0.8695  
Epoch 09 | train_loss=0.0417 train_acc=0.9870 | val_loss=0.4830 val_acc=0.8942  
  ✓ Saved best (val_acc=0.8942)
Epoch 10 | train_loss=0.0242

Training

In [55]:
import os
ckpt_dir = "/kaggle/working/checkpoints"
for f in os.listdir(ckpt_dir):
    size = os.path.getsize(os.path.join(ckpt_dir, f))
    print(f"  {f}  ({size/1024/1024:.1f} MB)")

  best_model.pt  (23.8 MB)
  spk2idx.json  (0.0 MB)
  training_log.json  (0.0 MB)


Kiểm tra output

In [56]:
!python evaluate_sid.py \
    --ckpt /kaggle/working/checkpoints/best_model.pt \
    --spk2idx /kaggle/working/checkpoints/spk2idx.json \
    --data_root /kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian \
    --split_file /kaggle/working/iden_split.txt

Loaded checkpoint từ epoch 12
Test samples: 730
Test: 100%|█████████████████████████████████████| 12/12 [00:06<00:00,  1.81it/s]

Speaker Identification Results
  Test samples:    730
  Top-1 accuracy:  92.05%
  Top-5 accuracy:  98.08%


Đánh giá SID

In [57]:
!python evaluate_sv.py \
    --ckpt /kaggle/working/checkpoints/best_model.pt \
    --data_root /kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian \
    --trial_file /kaggle/working/veri_test.txt

Loaded checkpoint từ epoch 12 (val_acc=0.9231)
Trials: 552 | Unique files: 584
Score trials: 100%|███████████████████████| 552/552 [00:00<00:00, 472212.08it/s]

Speaker Verification Results
  EER:                2.72%
  Decision threshold: 0.4064
  minDCF (p=0.01):    0.1014
Đã lưu kết quả vào sv_results.json


Đánh giá SV

In [58]:
import shutil
shutil.make_archive("/kaggle/working/checkpoints_export", "zip", "/kaggle/working/checkpoints")
print("Tải file checkpoints_export.zip từ tab Output của Kaggle")

Tải file checkpoints_export.zip từ tab Output của Kaggle


Download checkpoints